# Day 85: Synthetic Control: the Weight Problem

*Beginner to Ninja: Math for Causal Inference*

With one treated unit (a state, a country, a city) and many untreated ones, build a weighted average of untreated units that tracks the treated unit before the policy, and use it as the counterfactual afterwards. The constrained least-squares problem, why the weights are non-negative and sum to one, and when the method beats DiD.

**Learning outcomes**

- Formulate synthetic control as a constrained least-squares problem over the simplex and solve small cases.
- Compute synthetic outcomes, effect paths and pre-treatment fit (RMSPE).
- Explain the factor-model justification and why convex weights prevent extrapolation.
- Apply the method to tobacco control, reunification and soda taxes, and spot overfitting.

## Theory

### Setting

Unit 1 is treated from period $T_0 + 1$; units $2, \dots, J + 1$ (the **donor pool**) are never treated. With a single treated unit, standard errors from DiD are unreliable and a simple average of donors may be a poor comparison.

### The estimator (Abadie, Diamond and Hainmueller)

Choose donor weights $w = (w_2, \dots, w_{J+1})$ to solve

$$
\min_w\;\big\|X_1 - X_0w\big\|_V^2 \quad \text{s.t.} \quad w_j \ge 0,\; \sum_j w_j = 1,
$$

where $X_1$ stacks the treated unit's pre-treatment outcomes and predictors, $X_0$ the donors', and $V$ weights the predictors. The **synthetic control** outcome is $\hat Y_{1t}(0) = \sum_j w_jY_{jt}$ and the effect path

$$
\hat\tau_t = Y_{1t} - \sum_j w_jY_{jt}, \qquad t > T_0.
$$

### Why the constraints?

- **No extrapolation:** the synthetic unit lies in the convex hull of donors; with negative weights or weights summing to more than one, regression-type weights can fit the past by extrapolating and fail later.
- **Sparsity and transparency:** typically only a few donors get positive weight, and each can be inspected.
- Weights summing to one remove any common time effect.

### When does it work?

Suppose untreated outcomes follow a **linear factor model**

$$
Y_{it}(0) = \delta_t + \lambda_t^\top\mu_i + \varepsilon_{it},
$$

with unobserved loadings $\mu_i$ and time-varying factors $\lambda_t$. DiD assumes $\lambda_t^\top\mu_i$ reduces to a unit fixed effect. If the synthetic unit reproduces the treated unit's pre-period path over a long $T_0$, it approximately matches its loadings $\mu_1$, so it keeps tracking it afterwards: the bias is bounded and shrinks as $T_0$ grows relative to the noise. Good pre-treatment fit is necessary; with a short pre-period and many donors, a good fit can be overfitting noise.

### Practice

Exclude donors affected by the policy or by similar shocks; use predictors measured before treatment; report weights, pre-period fit, and the effect path; check robustness by leaving out donors. Inference is Day 86's topic. Extensions: augmented SC (bias correction with an outcome model), penalised SC, SC with multiple treated units.

## Part 1 · Warm-up

### Exercise 1 · A synthetic outcome

Weights 0.5, 0.3, 0.2 on three donor states whose outcomes in a post-period are 60, 80, 70. The treated state's outcome is 55. Compute the synthetic outcome and the effect.

<details><summary>Show solution</summary>

**Step 1.** $0.5(60) + 0.3(80) + 0.2(70) = 30 + 24 + 14 = 68$.

**Step 2.** $\hat\tau = 55 - 68 = -13$.

</details>

*Your work:*

### Exercise 2 · Pre-treatment fit

Pre-period gaps between treated and synthetic: 0.5, −0.3, 0.2, −0.4. Compute the RMSPE.

<details><summary>Show solution</summary>

**Step 1.** Squares: 0.25, 0.09, 0.04, 0.16; mean 0.135.

**Step 2.** RMSPE $= \sqrt{0.135} \approx 0.37$.

</details>

*Your work:*

### Exercise 3 · Two donors, one period

Treated pre-period outcome 5; donors 3 and 8. Find the weight $w$ on the first donor.

<details><summary>Show solution</summary>

**Step 1.** Solve $3w + 8(1 - w) = 5$: $8 - 5w = 5$, $w = 0.6$.

**Answer:** 0.6 and 0.4: an exact fit inside the hull.

</details>

*Your work:*

### Exercise 4 · Two donors, two periods

Treated (5, 6); donors (3, 4) and (8, 10). Find the least-squares weight on the first donor (weights sum to one).

<details><summary>Show solution</summary>

**Step 1.** Write the synthetic as $b + w(a - b)$ with $a - b = (-5, -6)$ and residual target $y_1 - b = (-3, -4)$.

**Step 2.** $w = \dfrac{(-3)(-5) + (-4)(-6)}{25 + 36} = \dfrac{39}{61} \approx 0.639$, inside $[0, 1]$.

</details>

*Your work:*

### Exercise 5 · Outside the hull

Treated pre-period outcome 9; donors 3 and 8. What happens?

<details><summary>Show solution</summary>

**Step 1.** Any convex combination lies in [3, 8].

**Answer:** the best feasible fit puts all weight on 8, leaving a pre-period gap of 1. Poor fit warns that the donors cannot reproduce the treated unit; the method should not be trusted without adjustment.

</details>

## Part 2 · Proofs

*Your work:*

### Exercise 6 · Weights summing to one remove time effects

Show that with $\sum_j w_j = 1$, a common shock $\delta_t$ cancels in $\hat\tau_t$.

<details><summary>Show solution</summary>

**Step 1.** $\sum_j w_j(\delta_t + \cdots) = \delta_t + \sum_j w_j(\cdots)$.

**Step 2.** The treated unit also contains $\delta_t$; the difference cancels it. $\blacksquare$

</details>

*Your work:*

### Exercise 7 · Unbiasedness with fixed effects and perfect fit

Let $Y_{it}(0) = \alpha_i + \delta_t + \varepsilon_{it}$ (no other factors, mean-zero noise) and suppose the weights satisfy $\sum_j w_j\alpha_j = \alpha_1$. Show $E[\hat\tau_t] = \tau_t$.

<details><summary>Show solution</summary>

**Step 1.** $\hat\tau_t = \tau_t + \alpha_1 + \delta_t + \varepsilon_{1t} - \sum w_j(\alpha_j + \delta_t + \varepsilon_{jt})$.

**Step 2.** $\delta_t$ cancels (Exercise 6) and $\alpha_1 - \sum w_j\alpha_j = 0$.

**Step 3.** Expectation of the noise terms is zero. $\blacksquare$

</details>

*Your work:*

### Exercise 8 · No extrapolation

Show that $\min_jY_{jt} \le \sum_jw_jY_{jt} \le \max_jY_{jt}$ for every $t$.

<details><summary>Show solution</summary>

**Step 1.** A convex combination is a weighted average with non-negative weights summing to one.

**Step 2.** It cannot exceed the largest term or fall below the smallest. $\blacksquare$ Regression weights lack this property.

</details>

*Your work:*

### Exercise 9 · Regression weights can be negative

Treated pre-period path (4, 6); donors (2, 2) and (3, 4). Find unconstrained weights (not summing to one, no intercept) that fit exactly.

<details><summary>Show solution</summary>

**Step 1.** Solve $2a + 3b = 4$ and $2a + 4b = 6$: $b = 2$, $a = -1$.

**Answer:** weights −1 and 2 fit perfectly but extrapolate: the synthetic unit is "twice donor 2 minus donor 1", which can behave wildly after treatment. $\blacksquare$

</details>

*Your work:*

### Exercise 10 · The two-donor solution in general

With two donors and weight $w$ on donor $a$, show the least-squares solution is the projection $w^* = \frac{(y_1 - b)^\top(a - b)}{\|a - b\|^2}$, clipped to $[0, 1]$.

<details><summary>Show solution</summary>

**Step 1.** Objective: $\|y_1 - b - w(a - b)\|^2$, a parabola in $w$.

**Step 2.** Unconstrained minimiser: set the derivative $-2(a - b)^\top(y_1 - b - w(a - b)) = 0$.

**Step 3.** A convex parabola restricted to $[0, 1]$ is minimised at the clipped value. $\blacksquare$

</details>

## Part 3 · Applied

*Your work:*

### Exercise 11 · California's tobacco programme

In a stylised version of the classic study, synthetic California puts weights 0.334, 0.234, 0.199, 0.164, 0.069 on Utah, Nevada, Montana, Colorado and Connecticut. In 2000 their per-capita cigarette sales (packs) are 40, 85, 75, 70 and 72; California's are 41.6. Compute the effect.

<details><summary>Show solution</summary>

**Step 1.** Synthetic: $0.334(40) + 0.234(85) + 0.199(75) + 0.164(70) + 0.069(72)$.

**Step 2.** $= 13.36 + 19.89 + 14.93 + 11.48 + 4.97 = 64.62$.

**Step 3.** $\hat\tau = 41.6 - 64.6 = -23.0$ packs per person.

**Answer:** the programme cut sales by about 23 packs per capita by 2000 (stylised figures).

</details>

*Your work:*

### Exercise 12 · German reunification

To estimate reunification's effect on West German GDP per capita, which donors and predictors would you use, and which would you exclude?

<details><summary>Show solution</summary>

**Step 1.** Donors: OECD countries not directly affected (e.g., Austria, US, Japan, Switzerland, Netherlands).

**Step 2.** Predictors: pre-1990 GDP per capita path, trade openness, inflation, schooling, industry share, investment rate.

**Step 3.** Exclude countries hit by the same shock (other countries absorbing large migration or affected by German unification directly) and use only pre-1990 values of predictors.

</details>

*Your work:*

### Exercise 13 · SC versus DiD in a simulation

In the simulation below, untreated outcomes follow a two-factor model; the treated region loads heavily on a trending factor, as do a few donors. The true effect is −2 after period 15. Compare synthetic control and DiD with all donors.

<details><summary>Show solution</summary>

**Step 1.** DiD assumes parallel trends with the average donor; the treated region's trending factor violates it, and DiD even gets the sign wrong (about +1.6).

**Step 2.** SC puts weight on the donors with similar loadings, fits the pre-period closely (RMSPE about 0.16) and estimates about −1.9.

</details>

*Your work:*

### Exercise 14 · Too many donors, too few periods

A city's homicide rate is matched with 80 donor cities over 4 pre-treatment years, with an excellent fit. Why be cautious?

<details><summary>Show solution</summary>

**Step 1.** With 80 donors and 4 periods, some combination fits almost any path, including its noise.

**Step 2.** A good pre-fit then says little about matching the underlying factors.

**Answer:** use more pre-periods or fewer, more similar donors; check fit on held-out pre-periods (backdating the placebo treatment).

</details>

*Your work:*

### Exercise 15 · A soda tax

A city introduces a tax on sugary drinks. Design a synthetic control analysis of beverage sales.

<details><summary>Show solution</summary>

**Step 1.** Outcome: monthly sales volume per capita of taxed beverages from scanner data.

**Step 2.** Donors: comparable cities without the tax, excluding neighbouring areas where cross-border shopping could raise sales (spillovers).

**Step 3.** Predictors: pre-tax sales paths, demographics, prices.

**Step 4.** Report weights, pre-period fit, the effect path, and a separate analysis of sales in neighbouring areas to measure cross-border substitution.

</details>

## Verify in Python

*Your work:*

In [ ]:
import numpy as np
from scipy.optimize import minimize

print("Ex 1:", .5 * 60 + .3 * 80 + .2 * 70, " Ex 2 RMSPE:", round(np.sqrt(np.mean(np.array([.5, -.3, .2, -.4]) ** 2)), 3))
a, b, y = np.array([3, 4.]), np.array([8, 10.]), np.array([5, 6.])
print("Ex 4 weight:", round((y - b) @ (a - b) / ((a - b) @ (a - b)), 4), " Ex 9:", np.linalg.solve([[2, 3], [2, 4]], [4, 6]))
w = np.array([.334, .234, .199, .164, .069]); print("Ex 11 synthetic:", round(w @ [40, 85, 75, 70, 72], 2), " effect:", round(41.6 - w @ [40, 85, 75, 70, 72], 2))

# Exercise 13: factor model, SC vs DiD
rng = np.random.default_rng(85)
J, T0, T = 20, 15, 25
mu = rng.normal(size=(J + 1, 2)); mu[1:4, 1] = [1.8, 1.4, 1.6]
lam = rng.normal(size=(T, 2)); lam[:, 1] = np.linspace(-2, 2, T)
alpha = rng.normal(0, 2, J + 1)
mu[0] = .5 * mu[1] + .3 * mu[2] + .2 * mu[3]; alpha[0] = .5 * alpha[1] + .3 * alpha[2] + .2 * alpha[3]
Y = alpha[:, None] + np.linspace(0, 3, T)[None, :] + mu @ lam.T + .2 * rng.normal(size=(J + 1, T))
Y[0, T0:] += -2.0
y1, Yd = Y[0, :T0], Y[1:, :T0]
res = minimize(lambda w: ((y1 - w @ Yd) ** 2).sum(), np.full(J, 1 / J), bounds=[(0, 1)] * J,
               constraints=({"type": "eq", "fun": lambda w: w.sum() - 1},), method="SLSQP")
w = res.x; gap = Y[0] - w @ Y[1:]
print("donors with weight > 0.02:", {j + 2: round(float(v), 2) for j, v in enumerate(w) if v > .02})
print(f"SC effect {gap[T0:].mean():.2f}  pre-RMSPE {np.sqrt((gap[:T0] ** 2).mean()):.2f}  "
      f"DiD {(Y[0, T0:].mean() - Y[0, :T0].mean()) - (Y[1:, T0:].mean() - Y[1:, :T0].mean()):.2f}  (truth -2)")

## Key takeaways

- Synthetic control builds a convex combination of donors that reproduces the treated unit's pre-treatment path and uses it as the counterfactual.
- Non-negative weights summing to one prevent extrapolation, remove common shocks and keep the comparison transparent.
- It is justified by factor models with a long pre-period; a good fit with few periods and many donors may be overfitting.